# Ch01-05: 데이터 변환과 정규화 이론


## 학습 목표

- Box-Cox 변환의 MLE 기반 최적 λ 추정 원리를 유도한다
- Yeo-Johnson 변환의 음수 확장 수학을 이해한다
- 분산 안정화 변환의 델타 방법 기반 이론을 이해한다
- Robust Scaling의 통계적 근거와 이상치 저항성을 분석한다


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

## 이론적 배경


### Box-Cox: $y^{(\lambda)}=\frac{y^\lambda-1}{\lambda}$ ($\lambda\neq0$), $\ln y$ ($\lambda=0$)

프로파일 우도: $\ell(\lambda)=-\frac{n}{2}\ln\hat{\sigma}^2(\lambda)+(\lambda-1)\sum\ln y_i$

### Yeo-Johnson: 음수 포함 확장

### 분산 안정화 (Delta Method)

$\text{Var}(Y)=g(\mu)$이면 $h'(\mu)\propto 1/\sqrt{g(\mu)}$

| 분포 | $g(\mu)$ | $h(y)$ |
|------|----------|--------|
| 포아송 | $\mu$ | $\sqrt{y}$ |
| 이항 | $\mu(1-\mu)$ | $\arcsin\sqrt{y}$ |

### Robust Scaling: $z=(x-\text{median})/\text{IQR}$, 붕괴점 25%


## 구현 가이드

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.preprocessing import PowerTransformer, RobustScaler

np.random.seed(42)
data_ln = np.random.lognormal(2, 1, 500)

def boxcox_ll(lam, y):
    n=len(y); yt = np.log(y) if abs(lam)<1e-10 else (y**lam-1)/lam
    s2=np.var(yt); return -(- n/2*np.log(s2)+(lam-1)*np.sum(np.log(y))) if s2>0 else 1e10

res = minimize_scalar(boxcox_ll, bounds=(-2,2), method='bounded', args=(data_ln,))
print(f"최적 λ: {res.x:.4f}")

fig, axes = plt.subplots(1,3,figsize=(15,4))
axes[0].hist(data_ln, bins=40, density=True); axes[0].set_title(f'원본 (skew={stats.skew(data_ln):.2f})')
bc, lam = stats.boxcox(data_ln)
axes[1].hist(bc, bins=40, density=True, color='orange'); axes[1].set_title(f'Box-Cox (λ={lam:.2f})')
yj = PowerTransformer('yeo-johnson').fit_transform(data_ln.reshape(-1,1)).flatten()
axes[2].hist(yj, bins=40, density=True, color='green'); axes[2].set_title(f'Yeo-Johnson')
plt.tight_layout(); plt.show()


---
## 연습 문제

### 문제 1 [★]

Box-Cox 프로파일 우도를 λ∈[-3,3]에서 그리고, 95% CI를 구하라 (우도비 기준).

In [ ]:
# TODO: 프로파일 우도 + CI


### 문제 2 [★★]

분산 안정화 변환을 델타 방법으로 유도하고 시뮬레이션 검증. 포아송(√y), 이항(arcsin√p).

In [ ]:
# TODO: 시뮬레이션


### 문제 3 [★★]

이상치 존재 시 Standard/MinMax/Robust/Power 스케일러 비교. 오염율별 분류 정확도.

In [ ]:
# TODO: 오염율별 실험


### 문제 4 [★★★]

Box-Cox λ의 점근 분산을 Fisher 정보량으로 유도, 부트스트랩(percentile, BCa)과 비교.

In [ ]:
# TODO: Fisher info + bootstrap CI


---
## 참고 자료

- Box & Cox (1964). An Analysis of Transformations. JRSS-B.
- Yeo & Johnson (2000). A New Family of Power Transformations.
